# Bước 3: Data Analysis with SQL
## UK Weather & Health: Weekly Deaths and Temperature (2016–2024)
---

## 3.1 Setup — Load dữ liệu vào SQLite

In [1]:
import pandas as pd
import sqlite3

# Load dataset đã xử lý từ Notebook 2
df = pd.read_csv('uk_weather_health_weekly_CLEANED.csv')

# Tạo SQLite database trong bộ nhớ
conn = sqlite3.connect(':memory:')
df.to_sql('weekly_health', conn, index=False, if_exists='replace')

print(f'Bảng "weekly_health" đã tạo với {len(df)} bản ghi.')
print(f'Các cột: {list(df.columns)}')

# Hàm trợ giúp chạy SQL
def sql(query):
    return pd.read_sql_query(query, conn)

Bảng "weekly_health" đã tạo với 442 bản ghi.
Các cột: ['week_start_sat', 'week_start_mon', 'week_end', 'deaths_total', 'tasmax_mean_weekly_degC', 'tasmax_max_weekly_degC', 'year', 'month', 'week_of_year', 'quarter', 'season', 'temp_range_weekly', 'IsSurge', 'tasmax_mean_scaled', 'tasmax_max_scaled']


In [2]:
# Kiểm tra cấu trúc bảng
sql("PRAGMA table_info(weekly_health)")

,cid,name,type,notnull,dflt_value,pk
0,0,week_start_sat,TEXT,0,None,0
1,1,week_start_mon,TEXT,0,None,0
2,2,week_end,TEXT,0,None,0
3,3,deaths_total,REAL,0,None,0
4,4,tasmax_mean_weekly_degC,REAL,0,None,0
5,5,tasmax_max_weekly_degC,REAL,0,None,0
6,6,year,INTEGER,0,None,0
7,7,month,INTEGER,0,None,0
8,8,week_of_year,INTEGER,0,None,0
9,9,quarter,INTEGER,0,None,0


## 3.2 Tổng số ca tử vong theo từng năm

In [3]:
sql("""
SELECT 
    year AS 'Năm',
    COUNT(*) AS 'Số tuần',
    ROUND(SUM(deaths_total), 0) AS 'Tổng tử vong',
    ROUND(AVG(deaths_total), 0) AS 'TB tử vong/tuần'
FROM weekly_health
GROUP BY year
ORDER BY year
""")

,Năm,Số tuần,Tổng tử vong,TB tử vong/tuần
0,2016,45,456934.0,10154.0
1,2017,46,474164.0,10308.0
2,2018,53,550295.0,10383.0
3,2019,41,413871.0,10094.0
4,2020,52,601860.0,11574.0
5,2021,49,550279.0,11230.0
6,2022,52,576896.0,11094.0
7,2023,52,581298.0,11179.0
8,2024,52,563732.0,10841.0


## 3.3 Năm nào có nhiệt độ trung bình cao nhất?

In [4]:
sql("""
SELECT 
    year AS 'Năm',
    ROUND(AVG(tasmax_mean_weekly_degC), 2) AS 'TB nhiệt độ (°C)',
    ROUND(MAX(tasmax_max_weekly_degC), 2) AS 'Nhiệt độ max (°C)'
FROM weekly_health
GROUP BY year
ORDER BY AVG(tasmax_mean_weekly_degC) DESC
""")

,Năm,TB nhiệt độ (°C),Nhiệt độ max (°C)
0,2022,13.98,31.36
1,2023,13.76,27.12
2,2020,13.47,27.71
3,2019,13.47,29.38
4,2024,13.42,24.29
5,2018,13.24,27.26
6,2021,13.21,26.51
7,2017,13.11,25.13
8,2016,12.20,22.84


## 3.4 Top 10 tuần có số ca tử vong cao nhất

In [5]:
sql("""
SELECT 
    week_start_mon AS 'Tuần bắt đầu',
    week_end AS 'Tuần kết thúc',
    CAST(deaths_total AS INTEGER) AS 'Số tử vong',
    ROUND(tasmax_mean_weekly_degC, 2) AS 'TB nhiệt độ (°C)',
    season AS 'Mùa',
    year AS 'Năm',
    IsSurge
FROM weekly_health
ORDER BY deaths_total DESC
LIMIT 10
""")

,Tuần bắt đầu,Tuần kết thúc,Số tử vong,TB nhiệt độ (°C),Mùa,Năm,IsSurge
0,2020-04-13,2020-04-17,22351,14.41,Spring,2020,1
1,2020-04-20,2020-04-24,21997,15.96,Spring,2020,1
2,2021-01-18,2021-01-22,18676,6.67,Winter,2021,1
3,2020-04-06,2020-04-10,18516,15.89,Spring,2020,1
4,2021-01-25,2021-01-29,18448,5.45,Winter,2021,1
5,2021-01-11,2021-01-15,18042,5.57,Winter,2021,1
6,2020-04-27,2020-05-01,17953,12.95,Spring,2020,1
7,2021-01-04,2021-01-08,17751,2.85,Winter,2021,1
8,2023-01-09,2023-01-13,17381,8.87,Winter,2023,1
9,2021-02-01,2021-02-05,17192,4.64,Winter,2021,1


## 3.5 Trung bình số ca tử vong theo từng mùa (Xuân, Hạ, Thu, Đông)

In [6]:
sql("""
SELECT 
    season AS 'Mùa',
    COUNT(*) AS 'Số tuần',
    ROUND(AVG(deaths_total), 0) AS 'TB tử vong/tuần',
    CAST(MIN(deaths_total) AS INTEGER) AS 'Min',
    CAST(MAX(deaths_total) AS INTEGER) AS 'Max',
    ROUND(AVG(tasmax_mean_weekly_degC), 2) AS 'TB nhiệt độ (°C)'
FROM weekly_health
GROUP BY season
ORDER BY AVG(deaths_total) DESC
""")

,Mùa,Số tuần,TB tử vong/tuần,Min,Max,TB nhiệt độ (°C)
0,Winter,113,12159.0,7131,18676,7.59
1,Spring,107,10948.0,6825,22351,12.62
2,Autumn,111,10418.0,8751,12535,13.62
3,Summer,111,9616.0,7739,11742,19.59


## 3.6 Có bao nhiêu tuần có nhiệt độ trung bình > 25°C?

In [7]:
sql("""
SELECT 
    COUNT(*) AS 'Số tuần có TB > 25°C'
FROM weekly_health
WHERE tasmax_mean_weekly_degC > 25
""")

,Số tuần có TB > 25°C
0,2


In [8]:
# Chi tiết các tuần đó
sql("""
SELECT 
    week_start_mon AS 'Tuần bắt đầu',
    ROUND(tasmax_mean_weekly_degC, 2) AS 'TB nhiệt độ (°C)',
    ROUND(tasmax_max_weekly_degC, 2) AS 'Max nhiệt độ (°C)',
    CAST(deaths_total AS INTEGER) AS 'Số tử vong',
    year AS 'Năm'
FROM weekly_health
WHERE tasmax_mean_weekly_degC > 25
ORDER BY tasmax_mean_weekly_degC DESC
""")

,Tuần bắt đầu,TB nhiệt độ (°C),Max nhiệt độ (°C),Số tử vong,Năm
0,2021-07-19,25.45,26.51,9744,2021
1,2023-06-12,25.31,26.38,10700,2023


## 3.7 Top 5 tuần có nhiệt độ thấp nhất và số tử vong tương ứng

In [9]:
sql("""
SELECT 
    week_start_mon AS 'Tuần bắt đầu',
    ROUND(tasmax_mean_weekly_degC, 2) AS 'TB nhiệt độ (°C)',
    ROUND(tasmax_max_weekly_degC, 2) AS 'Max nhiệt độ (°C)',
    CAST(deaths_total AS INTEGER) AS 'Số tử vong',
    season AS 'Mùa',
    year AS 'Năm',
    IsSurge
FROM weekly_health
ORDER BY tasmax_mean_weekly_degC ASC
LIMIT 5
""")

,Tuần bắt đầu,TB nhiệt độ (°C),Max nhiệt độ (°C),Số tử vong,Mùa,Năm,IsSurge
0,2018-02-26,0.96,5.07,10854,Winter,2018,0
1,2022-12-12,1.28,2.68,12389,Winter,2022,1
2,2021-02-08,1.49,4.81,15354,Winter,2021,1
3,2019-01-28,2.47,2.47,11297,Winter,2019,0
4,2021-01-04,2.85,3.65,17751,Winter,2021,1


## 3.8 Số lượng tuần được phân loại là IsSurge = 1

In [10]:
sql("""
SELECT 
    IsSurge,
    CASE WHEN IsSurge = 1 THEN 'Surge (Bùng phát)' ELSE 'Normal (Bình thường)' END AS 'Phân loại',
    COUNT(*) AS 'Số tuần',
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM weekly_health), 1) AS 'Tỷ lệ (%)',
    ROUND(AVG(deaths_total), 0) AS 'TB tử vong/tuần',
    ROUND(AVG(tasmax_mean_weekly_degC), 2) AS 'TB nhiệt độ (°C)'
FROM weekly_health
GROUP BY IsSurge
ORDER BY IsSurge
""")

,IsSurge,Phân loại,Số tuần,Tỷ lệ (%),TB tử vong/tuần,TB nhiệt độ (°C)
0,0,Normal (Bình thường),331,74.9,10013.0,14.74
1,1,Surge (Bùng phát),111,25.1,13107.0,9.14


In [11]:
# IsSurge theo từng năm
sql("""
SELECT 
    year AS 'Năm',
    SUM(CASE WHEN IsSurge = 1 THEN 1 ELSE 0 END) AS 'Số tuần Surge',
    COUNT(*) AS 'Tổng tuần',
    ROUND(SUM(CASE WHEN IsSurge = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS 'Tỷ lệ Surge (%)'
FROM weekly_health
GROUP BY year
ORDER BY year
""")

,Năm,Số tuần Surge,Tổng tuần,Tỷ lệ Surge (%)
0,2016,2,45,4.4
1,2017,9,46,19.6
2,2018,12,53,22.6
3,2019,4,41,9.8
4,2020,20,52,38.5
5,2021,14,49,28.6
6,2022,18,52,34.6
7,2023,17,52,32.7
8,2024,15,52,28.8


## 3.9 Trung bình nhiệt độ theo từng tháng (2016–2024)

In [12]:
sql("""
SELECT 
    month AS 'Tháng',
    ROUND(AVG(tasmax_mean_weekly_degC), 2) AS 'TB nhiệt độ TB (°C)',
    ROUND(AVG(tasmax_max_weekly_degC), 2) AS 'TB nhiệt độ max (°C)',
    ROUND(MIN(tasmax_mean_weekly_degC), 2) AS 'Thấp nhất (°C)',
    ROUND(MAX(tasmax_mean_weekly_degC), 2) AS 'Cao nhất (°C)',
    ROUND(AVG(deaths_total), 0) AS 'TB tử vong/tuần'
FROM weekly_health
GROUP BY month
ORDER BY month
""")

,Tháng,TB nhiệt độ TB (°C),TB nhiệt độ max (°C),Thấp nhất (°C),Cao nhất (°C),TB tử vong/tuần
0,1,7.07,9.24,2.47,11.17,13637.0
1,2,7.83,10.09,0.96,11.90,11886.0
2,3,9.75,11.88,4.94,14.66,11005.0
3,4,12.18,14.70,7.48,16.14,11793.0
4,5,16.27,18.53,10.35,21.51,10016.0
5,6,18.74,20.84,14.24,25.31,9900.0
6,7,20.34,22.80,16.55,25.45,9603.0
7,8,19.70,21.85,16.54,24.60,9346.0
8,9,17.35,19.71,13.60,24.82,9833.0
9,10,13.56,15.45,9.15,17.14,10401.0


## 3.10 Các tuần có nhiệt độ tăng đột ngột > 5°C so với tuần trước

In [13]:
sql("""
WITH temp_change AS (
    SELECT 
        w1.week_start_mon,
        w1.tasmax_mean_weekly_degC AS temp_current,
        w2.tasmax_mean_weekly_degC AS temp_prev,
        ROUND(w1.tasmax_mean_weekly_degC - w2.tasmax_mean_weekly_degC, 2) AS temp_diff,
        w1.deaths_total,
        w1.season,
        w1.year
    FROM weekly_health w1
    JOIN weekly_health w2 ON w1.ROWID = w2.ROWID + 1
)
SELECT 
    week_start_mon AS 'Tuần',
    ROUND(temp_prev, 2) AS 'Nhiệt độ tuần trước (°C)',
    ROUND(temp_current, 2) AS 'Nhiệt độ tuần này (°C)',
    temp_diff AS 'Chênh lệch (°C)',
    CAST(deaths_total AS INTEGER) AS 'Số tử vong',
    season AS 'Mùa',
    year AS 'Năm'
FROM temp_change
WHERE temp_diff > 5
ORDER BY temp_diff DESC
""")

,Tuần,Nhiệt độ tuần trước (°C),Nhiệt độ tuần này (°C),Chênh lệch (°C),Số tử vong,Mùa,Năm
0,2022-12-19,1.28,8.60,7.31,14530,Winter,2022
1,2021-02-15,1.49,8.18,6.68,13809,Winter,2021
2,2020-04-06,9.30,15.89,6.59,18516,Spring,2020
3,2024-01-22,3.31,9.88,6.58,12732,Winter,2024
4,2023-09-04,18.25,24.82,6.57,10168,Autumn,2023
5,2018-06-25,17.59,24.12,6.54,9212,Summer,2018
6,2023-06-12,18.77,25.31,6.54,10700,Summer,2023
7,2021-05-31,13.86,19.99,6.14,7778,Spring,2021
8,2016-01-25,5.36,11.17,5.81,11317,Winter,2016
9,2018-04-16,10.36,16.14,5.78,11223,Spring,2018


## 3.11 So sánh tổng số tử vong giữa trước và sau 2020

In [14]:
sql("""
SELECT 
    CASE 
        WHEN year < 2020 THEN 'Trước 2020 (2016–2019)'
        ELSE 'Từ 2020 trở đi (2020–2024)'
    END AS 'Giai đoạn',
    COUNT(*) AS 'Số tuần',
    ROUND(SUM(deaths_total), 0) AS 'Tổng tử vong',
    ROUND(AVG(deaths_total), 0) AS 'TB tử vong/tuần',
    CAST(MAX(deaths_total) AS INTEGER) AS 'Max tử vong/tuần',
    ROUND(AVG(tasmax_mean_weekly_degC), 2) AS 'TB nhiệt độ (°C)'
FROM weekly_health
GROUP BY CASE 
    WHEN year < 2020 THEN 'Trước 2020 (2016–2019)'
    ELSE 'Từ 2020 trở đi (2020–2024)'
END
ORDER BY MIN(year)
""")

,Giai đoạn,Số tuần,Tổng tử vong,TB tử vong/tuần,Max tử vong/tuần,TB nhiệt độ (°C)
0,Trước 2020 (2016–2019),185,1895264.0,10245.0,15050,13.00
1,Từ 2020 trở đi (2020–2024),257,2874065.0,11183.0,22351,13.57


In [15]:
# Chi tiết hơn: chia 3 giai đoạn (Trước COVID / COVID / Sau COVID)
sql("""
SELECT 
    CASE 
        WHEN year < 2020 THEN '1. Trước COVID (2016–2019)'
        WHEN year BETWEEN 2020 AND 2021 THEN '2. COVID (2020–2021)'
        ELSE '3. Sau COVID (2022–2024)'
    END AS 'Giai đoạn',
    COUNT(*) AS 'Số tuần',
    ROUND(SUM(deaths_total), 0) AS 'Tổng tử vong',
    ROUND(AVG(deaths_total), 0) AS 'TB tử vong/tuần',
    CAST(MAX(deaths_total) AS INTEGER) AS 'Max tử vong/tuần',
    SUM(IsSurge) AS 'Số tuần Surge'
FROM weekly_health
GROUP BY CASE 
    WHEN year < 2020 THEN '1. Trước COVID (2016–2019)'
    WHEN year BETWEEN 2020 AND 2021 THEN '2. COVID (2020–2021)'
    ELSE '3. Sau COVID (2022–2024)'
END
ORDER BY MIN(year)
""")

,Giai đoạn,Số tuần,Tổng tử vong,TB tử vong/tuần,Max tử vong/tuần,Số tuần Surge
0,1. Trước COVID (2016–2019),185,1895264.0,10245.0,15050,27
1,2. COVID (2020–2021),101,1152139.0,11407.0,22351,34
2,3. Sau COVID (2022–2024),156,1721926.0,11038.0,17381,50


## 3.12 Cleanup

In [16]:
conn.close()
print(' Đã đóng kết nối SQLite.')

 Đã đóng kết nối SQLite.


---
*Trước: [2. Data Collection](./2.%20Data%20Collection%2C%20Understanding%2C%20and%20Preparation.ipynb)* | *Tiếp: [4. Data Analysis with Python](./4.%20Data%20Analysis%20with%20Python.ipynb)*